In [1]:
import sys

sys.argv = [
    "program"
]


In [ ]:
from huggingface_hub import login

HF_TOKEN = 'hf_'
login(token=HF_TOKEN)

In [4]:
# scripts/02_emotion_direction_extraction/1_dump_residual_aligned_sublayer_activations.py
# -*- coding: utf-8 -*-
"""
Extract residual-aligned sublayer activations for Qwen model
Dump Residual-Aligned Sublayer Activations

Using accepted samples from 01 folder for forward computation, save attention and MLP layer outputs 
after adding back to residual stream for emotion vector calculation, grouped by all 6 emotions

- Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/sev/accepted.jsonl
- Output: outputs/{model_name}/02_emotion_directions/residual_dump/attention/ and mlp/
"""

import os, json, time, traceback, argparse
from pathlib import Path
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from collections import defaultdict

# HF token (prioritize env var, otherwise use default login)
HF_TOKEN = os.environ.get('HF_TOKEN', None)
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    # If already logged in via huggingface-cli login, this will use saved token
    pass

# Emotion categories
EMOS6 = ["anger","sadness","happiness","fear","surprise","disgust"]

# ============== Hook class to capture attention layer outputs ==============
class AttentionHook:
    """
    Capture attention layer output (before residual) for each layer
    Adapted for Qwen architecture: model.model.layers[layer_idx].self_attn
    """
    def __init__(self, model):
        self.model = model
        self.hooks = []
        self.attention_outputs = {}  # layer_idx -> attention_output (before residual)
        self.register_hooks()
    
    def register_hooks(self):
        """
        Register hooks to attention layers
        Qwen specific: attention module is 'self_attn' and output[0] is the attention output
        """
        for layer_idx, layer in enumerate(self.model.model.layers):
            # Hook attention layer - at self_attn output
            def make_attention_hook(idx):
                def attention_hook(module, input, output):
                    # For Qwen attention, output is a tuple where output[0] is attention output (before residual)
                    if isinstance(output, tuple) and len(output) > 0:
                        attention_output = output[0]  # attention layer output (before residual)
                        self.attention_outputs[idx] = attention_output.detach().cpu()
                return attention_hook
            
            # Register hook
            attn_hook = layer.self_attn.register_forward_hook(make_attention_hook(layer_idx))
            self.hooks.append(attn_hook)
    
    def clear_outputs(self):
        """Clear output cache"""
        self.attention_outputs.clear()
    
    def remove_hooks(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

# ============== Forward computation function ==============
@torch.no_grad()
def forward_with_hooks(inputs, model, hook_manager):
    """
    Execute forward pass and capture attention layer outputs and MLP outputs (hidden states)
    Adapted for Qwen's hidden states structure
    """
    hook_manager.clear_outputs()
    
    # Execute forward pass
    outputs = model(**inputs, output_hidden_states=True)
    
    # Get last token index
    last_idx = inputs.input_ids.shape[1] - 1
    
    # Extract attention output at last token position for each layer (after residual)
    attention_outputs = {}
    for layer_idx in range(len(model.model.layers)):
        if layer_idx in hook_manager.attention_outputs:
            # Get output at last token position
            attn_output = hook_manager.attention_outputs[layer_idx][:, last_idx, :].squeeze(0)
            # Get residual input (layer input) from hidden states
            # hidden_states[layer_idx] is input to layer_idx
            residual = outputs.hidden_states[layer_idx][:, last_idx, :].squeeze(0).cpu()
            # Attention output plus residual
            attn_plus_residual = attn_output + residual
            attention_outputs[layer_idx] = attn_plus_residual.float().cpu()
    
    # MLP output after residual is hidden states (skip embedding layer, start from layer 1)
    mlp_outputs = {}
    for layer_idx in range(len(model.model.layers)):
        # hidden_states[0] is embedding output, hidden_states[1] is layer 0 output, etc.
        hidden_state = outputs.hidden_states[layer_idx + 1][:, last_idx, :].squeeze(0)
        mlp_outputs[layer_idx] = hidden_state.float().cpu()
    
    return attention_outputs, mlp_outputs, int(last_idx)

# ============== Read accepted samples ==============
def iter_accepted_samples(path):
    """Read samples from accepted.jsonl"""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                yield obj
            except json.JSONDecodeError:
                continue

# ============== Build messages ==============
def build_messages(emotion: str, scenario: str, event: str):
    """Build conversation messages"""
    system = f''' Always reply in {emotion}. Keep the reply to at most two sentences. '''.strip()
    user = f"{scenario}\n{event}"
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

def prepare_inputs(tok, messages, device):
    """Apply chat template and tensorize to model device"""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(device)
    return inputs

# ============== Save function ==============
def save_group_outputs(group_data, attention_outputs, mlp_outputs, last_idx, attention_save_dir, mlp_save_dir):
    """Save all emotion outputs for one group"""
    skeleton_id = group_data["skeleton_id"]
    valence = group_data["valence"]
    
    # Build base filename
    base = f"{skeleton_id}__{valence}"
    
    # Prepare attention output data
    attention_data = {
        "hidden_last_all_layers": {emotion: torch.stack([attention_outputs[emotion][i] for i in sorted(attention_outputs[emotion].keys())], dim=0).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},
        "logits_0": {emotion: torch.zeros(1).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},  # placeholder
        "input_ids": {emotion: torch.tensor([0], dtype=torch.int32) for emotion in EMOS6},  # placeholder
        "last_input_idx": {emotion: torch.tensor(last_idx, dtype=torch.int32) for emotion in EMOS6},
        "gen_text": {emotion: group_data["gen_texts"][emotion] for emotion in EMOS6}
    }
    
    # Prepare MLP output data
    mlp_data = {
        "hidden_last_all_layers": {emotion: torch.stack([mlp_outputs[emotion][i] for i in sorted(mlp_outputs[emotion].keys())], dim=0).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},
        "logits_0": {emotion: torch.zeros(1).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},  # placeholder
        "input_ids": {emotion: torch.tensor([0], dtype=torch.int32) for emotion in EMOS6},  # placeholder
        "last_input_idx": {emotion: torch.tensor(last_idx, dtype=torch.int32) for emotion in EMOS6},
        "gen_text": {emotion: group_data["gen_texts"][emotion] for emotion in EMOS6}
    }
    
    # Save attention data
    attention_path = attention_save_dir / f"{base}.pt"
    torch.save(attention_data, attention_path)
    
    # Save MLP data
    mlp_path = mlp_save_dir / f"{base}.pt"
    torch.save(mlp_data, mlp_path)
    
    return attention_path, mlp_path

# ============== Main process ==============
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="qwen3_4b_instruct_2507",
                       help="Model folder name")
    parser.add_argument("--model", type=str, default="Qwen/Qwen3-4B-Instruct-2507",
                       help="HuggingFace model name")
    parser.add_argument("--device", type=str, default="cuda:0",
                       help="Device")
    parser.add_argument("--dtype", type=str, default="float16", choices=["float32","bfloat16","float16"],
                       help="Data type")
    args = parser.parse_args()
    
    # Fixed to use sev dataset
    model_name = args.model_name
    dataset_name = "sev"
    
    # Build input path
    input_path = Path("/workspace/Various-Model/outputs") / model_name / "01_emotion_elicited_generation_prompt_based" / "labeled" / dataset_name / "accepted.jsonl"
    
    if not input_path.exists():
        print(f"[ERROR] Input file not found: {input_path}")
        return
    
    # Build output paths
    attention_save_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "attention"
    mlp_save_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "mlp"
    attention_save_dir.mkdir(parents=True, exist_ok=True)
    mlp_save_dir.mkdir(parents=True, exist_ok=True)
    
    # Data type mapping
    dmap = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}
    torch_dtype = dmap[args.dtype]
    
    # Load model and tokenizer
    print("Loading model and tokenizer...")
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, token=HF_TOKEN if HF_TOKEN else True, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        torch_dtype=torch_dtype,
        device_map=args.device,
        token=HF_TOKEN if HF_TOKEN else True,
        trust_remote_code=True
    )
    model.eval()
    
    print("Device:", model.device)
    print("Processing accepted samples...")
    
    # Create hook manager
    hook_manager = AttentionHook(model)
    
    started = time.time()
    processed = 0
    skipped = 0
    
    # Collect data by group
    group_data = defaultdict(lambda: {
        "skeleton_id": None,
        "valence": None,
        "attention_outputs": {},
        "mlp_outputs": {},
        "gen_texts": {},
        "last_idx": None
    })
    
    try:
        for sample in iter_accepted_samples(input_path):
            key = sample["key"]
            skeleton_id = sample["skeleton_id"]
            valence = sample["valence"]
            emotion = sample["emotion"]
            
            group_key = f"{skeleton_id}__{valence}"
            
            # Skip already processed groups
            attention_path = attention_save_dir / f"{group_key}.pt"
            mlp_path = mlp_save_dir / f"{group_key}.pt"
            
            if attention_path.exists() and mlp_path.exists():
                skipped += 1
                if skipped % 50 == 0:
                    print(f"[SKIP] {skipped} groups skipped so far... (last: {group_key})")
                continue
            
            try:
                # Build messages and inputs
                messages = build_messages(
                    sample["emotion"], 
                    sample["scenario"], 
                    sample["event"]
                )
                inputs = prepare_inputs(tok, messages, model.device)
                
                # Execute forward pass and capture outputs
                attention_outputs, mlp_outputs, last_idx = forward_with_hooks(inputs, model, hook_manager)
                
                # Collect into group data
                group_data[group_key]["skeleton_id"] = skeleton_id
                group_data[group_key]["valence"] = valence
                group_data[group_key]["attention_outputs"][emotion] = attention_outputs
                group_data[group_key]["mlp_outputs"][emotion] = mlp_outputs
                group_data[group_key]["gen_texts"][emotion] = sample["gen_text"]
                group_data[group_key]["last_idx"] = last_idx
                
                # Check if all 6 emotions have been collected
                if len(group_data[group_key]["attention_outputs"]) == 6:
                    # Save group data
                    save_group_outputs(group_data[group_key],
                                      group_data[group_key]["attention_outputs"],
                                      group_data[group_key]["mlp_outputs"],
                                      group_data[group_key]["last_idx"],
                                      attention_save_dir,
                                      mlp_save_dir)
                    
                    processed += 1
                    if processed % 10 == 0:
                        elapsed = time.time() - started
                        rate = processed / elapsed if elapsed > 0 else 0
                        print(f"[progress] processed={processed}, skipped={skipped}, elapsed={elapsed:.1f}s, rate={rate:.2f} groups/s")
                    
                    # Clean up processed group data
                    del group_data[group_key]
            
            except Exception as e:
                print(f"Error processing {key}: {e}")
                traceback.print_exc()
                continue
    
    finally:
        # Clean up hooks
        hook_manager.remove_hooks()
    
    elapsed = time.time() - started
    print(f"\n[OK] Done. processed={processed} groups, skipped={skipped}.")
    print(f"     Time: {elapsed:.1f}s | Rate: {processed/elapsed:.2f} groups/s")
    print(f"     Attention outputs saved to: {attention_save_dir}")
    print(f"     MLP outputs saved to: {mlp_save_dir}")
    print("     Note: Use next script to compute emotion directions from these saved states.")

if __name__ == "__main__":
    main()

Loading model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Device: cuda:0
Processing accepted samples...
[progress] processed=10, skipped=0, elapsed=2.0s, rate=5.06 groups/s
[progress] processed=20, skipped=0, elapsed=3.7s, rate=5.35 groups/s
[progress] processed=30, skipped=0, elapsed=5.5s, rate=5.46 groups/s
[progress] processed=40, skipped=0, elapsed=7.3s, rate=5.51 groups/s
[progress] processed=50, skipped=0, elapsed=9.0s, rate=5.54 groups/s
[progress] processed=60, skipped=0, elapsed=10.8s, rate=5.57 groups/s
[progress] processed=70, skipped=0, elapsed=12.7s, rate=5.52 groups/s
[progress] processed=80, skipped=0, elapsed=14.5s, rate=5.50 groups/s
[progress] processed=90, skipped=0, elapsed=16.3s, rate=5.53 groups/s
[progress] processed=100, skipped=0, elapsed=18.0s, rate=5.55 groups/s
[progress] processed=110, skipped=0, elapsed=20.0s, rate=5.49 groups/s
[progress] processed=120, skipped=0, elapsed=21.9s, rate=5.47 groups/s
[progress] processed=130, skipped=0, elapsed=24.3s, rate=5.34 groups/s
[progress] processed=140, skipped=0, elapsed=

In [5]:
import sys

sys.argv = [
    "program"
]


In [6]:
# scripts/02_emotion_direction_extraction/2_compute_emotion_directions.py
# -*- coding: utf-8 -*-
"""
Compute emotion direction vectors for Qwen model
Compute Emotion Direction Vectors

From residual_dump/attention/ and mlp/ compute emotion direction vectors

Method:
- Within each group: 6 emotions - group mean → de-contextualization
- Cross-group mean to get emotion centroids
- Remove global mean and normalize to get emotion direction vectors v_e^{(l)}
- Positions are different points in the residual stream

Input: outputs/{model_name}/02_emotion_directions/residual_dump/attention/ and mlp/
Output: outputs/{model_name}/02_emotion_directions/emo_directions_mlp.pt and emo_directions_attention.pt
"""

import os, argparse
from pathlib import Path
import numpy as np
import torch

# 6 emotions
EMOS6 = ["anger","sadness","happiness","fear","surprise","disgust"]

def load_groups(data_dir: Path, data_type: str):
    """
    Load data of specified type, return groups: list[dict emo->[L,H]]
    """
    pts = sorted(data_dir.glob("*.pt"))
    
    print(f"Loading {len(pts)} files for {data_type} emotion direction calculation...")
    
    # Group by skeleton_id and valence, collect all emotion data
    from collections import defaultdict
    grouped_data = defaultdict(lambda: defaultdict(dict))
    
    for p in pts:
        try:
            rec = torch.load(p, map_location="cpu")
        except Exception as e:
            print(f"[LOAD-ERROR] {p.name}: {e}")
            continue
        
        H = rec.get("hidden_last_all_layers", {})
        if not H:
            print(f"[SKIP] {p.name}: no hidden_last_all_layers")
            continue
        
        # Extract skeleton_id and valence from filename
        # Format: skeleton_id__valence.pt
        filename = p.stem
        if '__' not in filename:
            print(f"[SKIP] {p.name}: invalid filename format")
            continue
        
        skeleton_id, valence = filename.rsplit('__', 1)
        group_key = f"{skeleton_id}__{valence}"
        
        # Get emotion data from hidden_last_all_layers
        for emotion, tensor in H.items():
            if emotion in EMOS6:
                grouped_data[group_key][emotion] = tensor.numpy().astype(np.float32)
    
    # Convert to groups format, keep only groups with all 6 emotions
    groups = []
    for group_key, emotions in grouped_data.items():
        if all(e in emotions for e in EMOS6):
            groups.append({e: emotions[e] for e in EMOS6})
        else:
            missing = [e for e in EMOS6 if e not in emotions]
            print(f"[SKIP] {group_key}: missing emotions {missing}")
    
    if not groups:
        raise RuntimeError(f"No usable groups found in {data_dir}")
    
    L = next(iter(groups))[EMOS6[0]].shape[0]
    Hdim = next(iter(groups))[EMOS6[0]].shape[1]
    
    print(f"Loaded {len(groups)} groups for {data_type} | layers={L}, H={Hdim}")
    return groups, L, Hdim

def build_emotion_directions(groups, L, Hdim):
    """
    Compute emotion direction vectors
    
    Returns: dirs: dict emo->[L,H]
    """
    # Remove within-group mean
    centered = []
    for g in groups:
        mu = np.stack([g[e] for e in EMOS6], 0).mean(axis=0)  # [L,H]
        centered.append({e: (g[e] - mu).astype(np.float32) for e in EMOS6})
    
    # Cross-group emotion centroids
    centroids = {e: np.stack([cg[e] for cg in centered], 0).mean(axis=0) for e in EMOS6}  # emo->[L,H]
    
    # Remove global mean
    all_mean = np.stack(list(centroids.values()), 0).mean(axis=0)  # [L,H]
    
    # Normalize to get emotion direction vectors
    dirs = {}
    for e in EMOS6:
        v = centroids[e] - all_mean
        n = np.linalg.norm(v, axis=1, keepdims=True) + 1e-12  # [L,1]
        dirs[e] = (v / n).astype(np.float32)  # [L,H]
    return dirs

def compute_mlp_emotion_directions(mlp_dir, out_dir):
    """
    Compute MLP emotion direction vectors
    """
    print("\n=== Computing MLP Emotion Directions ===")
    
    try:
        groups, L, Hdim = load_groups(mlp_dir, "MLP")
        dirs = build_emotion_directions(groups, L, Hdim)
        
        # Save MLP emotion vectors
        out_path = out_dir / "emo_directions_mlp.pt"
        torch.save({
            "dirs": dirs, 
            "layers": L, 
            "hidden": Hdim, 
            "emotions": EMOS6,
            "type": "mlp",
            "model": "Qwen3-4B-Instruct"
        }, out_path)
        print(f"[✓] Saved MLP emotion directions to {out_path}")
        
        # Print verification
        print("MLP Emotion Directions:")
        for e in EMOS6:
            arr = dirs[e]
            print(f"  {e:>9s} | shape={arr.shape} | mean-norm={np.mean(np.linalg.norm(arr, axis=1)):.4f}")
        
        return True
    
    except Exception as e:
        print(f"[ERROR] Failed to compute MLP emotion directions: {e}")
        return False

def compute_attention_emotion_directions(attention_dir, out_dir):
    """
    Compute Attention emotion direction vectors
    """
    print("\n=== Computing Attention Emotion Directions ===")
    
    try:
        groups, L, Hdim = load_groups(attention_dir, "Attention")
        dirs = build_emotion_directions(groups, L, Hdim)
        
        # Save Attention emotion vectors
        out_path = out_dir / "emo_directions_attention.pt"
        torch.save({
            "dirs": dirs, 
            "layers": L, 
            "hidden": Hdim, 
            "emotions": EMOS6,
            "type": "attention",
            "model": "Qwen3-4B-Instruct"
        }, out_path)
        print(f"[✓] Saved Attention emotion directions to {out_path}")
        
        # Print verification
        print("Attention Emotion Directions:")
        for e in EMOS6:
            arr = dirs[e]
            print(f"  {e:>9s} | shape={arr.shape} | mean-norm={np.mean(np.linalg.norm(arr, axis=1)):.4f}")
        
        return True
    
    except Exception as e:
        print(f"[ERROR] Failed to compute Attention emotion directions: {e}")
        return False

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="qwen3_4b_instruct_2507",
                       help="Model folder name")
    args = parser.parse_args()
    
    print("Computing emotion directions for MLP and Attention (Qwen3-4B-Instruct)")
    print("=" * 60)
    
    # Build paths
    model_name = args.model_name
    attention_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "attention"
    mlp_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "mlp"
    out_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions"
    
    # Check if input directories exist
    if not attention_dir.exists():
        print(f"[ERROR] Attention directory not found: {attention_dir}")
        return
    
    if not mlp_dir.exists():
        print(f"[ERROR] MLP directory not found: {mlp_dir}")
        return
    
    # Compute MLP emotion directions
    mlp_success = compute_mlp_emotion_directions(mlp_dir, out_dir)
    
    # Compute Attention emotion directions
    attention_success = compute_attention_emotion_directions(attention_dir, out_dir)
    
    # Summary
    print("\n" + "=" * 60)
    print("SUMMARY:")
    print(f"MLP emotion directions: {'✓ SUCCESS' if mlp_success else '✗ FAILED'}")
    print(f"Attention emotion directions: {'✓ SUCCESS' if attention_success else '✗ FAILED'}")
    
    if mlp_success and attention_success:
        print(f"\nAll emotion directions saved to: {out_dir}")
        print("- emo_directions_mlp.pt: MLP-based emotion directions")
        print("- emo_directions_attention.pt: Attention-based emotion directions")
    else:
        print("\nSome computations failed. Check the error messages above.")

if __name__ == "__main__":
    main()

Computing emotion directions for MLP and Attention (Qwen3-4B-Instruct)

=== Computing MLP Emotion Directions ===
Loading 405 files for MLP emotion direction calculation...
Loaded 405 groups for MLP | layers=36, H=2560
[✓] Saved MLP emotion directions to /workspace/Various-Model/outputs/qwen3_4b_instruct_2507/02_emotion_directions/emo_directions_mlp.pt
MLP Emotion Directions:
      anger | shape=(36, 2560) | mean-norm=1.0000
    sadness | shape=(36, 2560) | mean-norm=1.0000
  happiness | shape=(36, 2560) | mean-norm=1.0000
       fear | shape=(36, 2560) | mean-norm=1.0000
   surprise | shape=(36, 2560) | mean-norm=1.0000
    disgust | shape=(36, 2560) | mean-norm=1.0000

=== Computing Attention Emotion Directions ===
Loading 405 files for Attention emotion direction calculation...
Loaded 405 groups for Attention | layers=36, H=2560
[✓] Saved Attention emotion directions to /workspace/Various-Model/outputs/qwen3_4b_instruct_2507/02_emotion_directions/emo_directions_attention.pt
Attention

In [10]:
import sys

sys.argv = [
    "program",
    "--dataset_name", "test_set",
    "--has_valence"
]


In [11]:
# scripts/03_emotion_elicited_generation_steer_based/1_steer_with_emotion_direction_qwen.py
# -*- coding: utf-8 -*-
"""
Emotion-Steered Generation Script - Using Emotion Vectors to Guide Generation (Adapted for Qwen3-4B-Instruct)

Uses extracted emotion direction vectors to guide generation for input data.

Default parameters for Qwen (36 layers):
- layers: 15-25 (Layers to apply steering - middle layers)
- alpha: 8 (Steering strength)
- last_k: 1 (Number of positions to steer)
- scale: rms (Scaling mode)
- dtype: float32 (Data type)
- direction_type: mlp (Direction type: mlp or attention)

Input: data/{dataset_name}.jsonl
Output: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/steered_outputs.jsonl
"""

import os, json, argparse, time
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# HF token (prioritize env var, otherwise use default login)
HF_TOKEN = os.environ.get('HF_TOKEN', None)
if HF_TOKEN:
    login(token=HF_TOKEN)

# 6 emotions and 3 valences
EMOS6 = ["anger","sadness","happiness","fear","surprise","disgust"]
VALENCES = ["positive", "neutral", "negative"]

# Qwen3-4B specific constants
QWEN_3B_LAYERS = 36
QWEN_3B_HIDDEN = 2560

def build_messages(scenario: str, event: str):
    """
    Build conversation messages using standard chat template
    Qwen uses standard chat template format
    """
    system = "Keep the reply to at most two sentences."
    user = f"{scenario}\n{event}"
    return [{"role":"system","content":system},{"role":"user","content":user}]

def load_directions(path: Path):
    """
    Load emotion direction vectors from saved .pt file
    """
    obj = torch.load(path, map_location="cpu", weights_only=False)
    
    if "dirs" in obj:
        dirs = obj["dirs"]
        metadata = obj.get("metadata", {})
        L = metadata.get("layers", QWEN_3B_LAYERS)
        H = metadata.get("hidden_dim", QWEN_3B_HIDDEN)
        emos = metadata.get("emotions", EMOS6)
    else:
        dirs = obj["dirs"]
        L = obj["layers"]
        H = obj["hidden"]
        emos = obj["emotions"]
    
    for e in dirs:
        if not isinstance(dirs[e], np.ndarray):
            dirs[e] = np.array(dirs[e], dtype=np.float32)
        else:
            dirs[e] = dirs[e].astype(np.float32)
    
    return dirs, L, H, emos

def parse_layers(arg: str):
    """
    Parse layer range (e.g., '15-25') or comma-separated list
    """
    if "-" in arg:
        a, b = arg.split("-")
        return list(range(int(a), int(b) + 1))
    return [int(x) for x in arg.split(",") if x.strip()]

class EmotionSteerer:
    """
    Hook manager to apply steering vectors to specific layers during forward pass
    Adapted for Qwen architecture: model.model.layers[layer_idx]
    """
    def __init__(self, model, dirs_np, target_emotion, layer_ids, alpha=8.0, last_k=1, scale_mode="rms"):
        self.model = model
        self.alpha = float(alpha)
        self.layer_ids = list(layer_ids)
        self.last_k = int(last_k)
        self.scale_mode = scale_mode
        
        # Qwen specific: layers are in model.model.layers
        if hasattr(model, 'model') and hasattr(model.model, 'layers'):
            self.layers = model.model.layers
        else:
            # Fallback search for layers
            self.layers = None
            for name, module in model.named_modules():
                if 'layers' in name and hasattr(module, '__len__'):
                    self.layers = module
                    break
        
        if self.layers is None:
            raise ValueError("Could not find transformer layers in the model")
        
        print(f"Found {len(self.layers)} layers in model")
        
        # Prepare steering vectors for specified layers
        self.v = {}
        for l in self.layer_ids:
            if l < len(self.layers):
                # Convert numpy array to torch tensor and move to device
                self.v[l] = torch.from_numpy(dirs_np[target_emotion][l]).to(model.device)
        
        # Register hooks
        self.handles = []
        for l in self.layer_ids:
            if l in self.v:
                h = self.layers[l].register_forward_hook(self._make_hook(l))
                self.handles.append(h)
    
    def _make_hook(self, layer_id: int):
        v = self.v[layer_id]
        alpha = self.alpha
        last_k = self.last_k
        scale_mode = self.scale_mode
        
        def hook(module, inputs, output):
            # Qwen layers typically return a tuple or tensor
            if isinstance(output, (tuple, list)):
                # For tuple output (like attention layers that return multiple values)
                if len(output) > 0:
                    hs = output[0].clone()  # First element is the hidden states
                    B, T, H = hs.shape
                    start = max(0, T - last_k)
                    
                    if scale_mode == "rms":
                        seg = hs[:, start:T, :]
                        rms = torch.sqrt((seg**2).mean(dim=-1, keepdim=True) + 1e-12)
                        delta = alpha * v.view(1, 1, H) * rms
                    else:  # unit scaling
                        delta = alpha * v.view(1, 1, H)
                    
                    hs[:, start:T, :] = hs[:, start:T, :] + delta
                    return (hs,) + output[1:]  # Return modified hidden states with other outputs
                return output
            else:
                # For tensor output (simpler layers)
                hs = output.clone()
                B, T, H = hs.shape
                start = max(0, T - last_k)
                
                if scale_mode == "rms":
                    seg = hs[:, start:T, :]
                    rms = torch.sqrt((seg**2).mean(dim=-1, keepdim=True) + 1e-12)
                    delta = alpha * v.view(1, 1, H) * rms
                else:  # unit scaling
                    delta = alpha * v.view(1, 1, H)
                
                hs[:, start:T, :] = hs[:, start:T, :] + delta
                return hs
        return hook
    
    def remove(self):
        """Remove all hooks"""
        for h in self.handles:
            h.remove()
        self.handles.clear()

@torch.no_grad()
def generate_text(model, tok, messages, max_new_tokens=500):
    """
    Generate response using greedy decoding
    """
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    
    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        use_cache=True,
        min_new_tokens=10,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
    )
    
    out_ids = gen[0][inputs.input_ids.shape[1]:]
    return tok.decode(out_ids, skip_special_tokens=True).strip()

def load_user_inputs(data_path: str, has_valence: bool = False):
    """
    Load JSONL input data
    """
    inputs = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: 
                continue
            try:
                obj = json.loads(line)
                if has_valence:
                    if "event" in obj and "scenario" in obj:
                        event_data = obj["event"]
                        if all(v in event_data for v in VALENCES):
                            inputs.append(obj)
                else:
                    if "event" in obj and "scenario" in obj:
                        inputs.append(obj)
            except json.JSONDecodeError:
                continue
    return inputs

def get_recommended_layers(model_type="qwen"):
    """
    Get recommended layers based on model architecture
    """
    if model_type == "qwen":
        # For 36 layers, use middle layers 15-25
        return "15-25"
    else:
        # Default range
        return "11-20"

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_name", type=str, default="qwen3_4b_instruct_2507",
                   help="Model folder name")
    ap.add_argument("--model", type=str, default="Qwen/Qwen3-4B-Instruct-2507",
                   help="HuggingFace model name")
    ap.add_argument("--dataset_name", type=str, default="test_set",
                   help="Dataset name")
    ap.add_argument("--direction_type", type=str, default="mlp", choices=["mlp", "attention"],
                   help="Type of emotion directions to use")
    ap.add_argument("--layers", type=str, default="15-25",
                   help="Layer range to apply steering (e.g., '15-25' for Qwen's 36 layers)")
    ap.add_argument("--alpha", type=float, default=8.0,
                   help="Steering strength")
    ap.add_argument("--last_k", type=int, default=1,
                   help="Number of last token positions to steer")
    ap.add_argument("--scale", type=str, default="rms", choices=["unit","rms"],
                   help="Scaling mode")
    ap.add_argument("--dtype", type=str, default="float16", choices=["float16","bfloat16","float32"],
                   help="Data type")
    ap.add_argument("--max_new_tokens", type=int, default=100,
                   help="Maximum new tokens to generate")
    ap.add_argument("--data_path", type=str, default=None,
                   help="Custom data path (optional)")
    ap.add_argument("--has_valence", action="store_true",
                   help="Whether input has valence dimension")
    ap.add_argument("--device", type=str, default="cuda:0",
                   help="Device")
    args = ap.parse_args()
    
    print("Emotion-Steered Generation Script (Qwen3-4B-Instruct)")
    print("=" * 60)
    print(f"Model: {args.model}")
    print(f"Layers: {args.layers} (total 36 layers in Qwen)")
    print(f"Alpha: {args.alpha}, Last_k: {args.last_k}, Scale: {args.scale}")
    print(f"Direction type: {args.direction_type}")
    print("=" * 60)
    
    # Paths
    model_name = args.model_name
    dataset_name = args.dataset_name
    directions_file = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / f"emo_directions_{args.direction_type}.pt"
    data_path = args.data_path if args.data_path else Path("/workspace/Various-Model/data") / f"{dataset_name}.jsonl"
    out_dir = Path("/workspace/Various-Model/outputs") / model_name / "03_emotion_steered_generation" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    output_path = out_dir / "steered_outputs.jsonl"
    
    # Check if directions file exists
    if not directions_file.exists():
        print(f"[ERROR] Direction file not found: {directions_file}")
        print(f"Please run the direction computation script first.")
        return
    
    # Load directions
    dirs, L, Hdim, emos = load_directions(directions_file)
    print(f"[directions] Loaded {len(emos)} emotions, {L} layers, hidden dim {Hdim}")
    
    # Check if layer range is valid
    layer_ids = parse_layers(args.layers)
    if max(layer_ids) >= L:
        print(f"[WARNING] Requested layers {args.layers} exceed available layers ({L})")
        print(f"Clipping to available layers...")
        layer_ids = [l for l in layer_ids if l < L]
    
    # Load dataset
    if not Path(data_path).exists():
        print(f"[ERROR] Data file not found: {data_path}")
        return
    
    user_inputs = load_user_inputs(data_path, has_valence=args.has_valence)
    print(f"[data] Loaded {len(user_inputs)} inputs from {data_path}")
    
    # Load model
    torch_dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[args.dtype]
    
    print("Loading Qwen3-4B-Instruct model...")
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, token=HF_TOKEN if HF_TOKEN else True, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        torch_dtype=torch_dtype,
        device_map=args.device,
        token=HF_TOKEN if HF_TOKEN else True,
        trust_remote_code=True
    )
    model.eval()
    print(f"Model loaded on {model.device}")
    
    # Steering parameters
    print(f"\n[steering] Applying to layers: {layer_ids}")
    print(f"[steering] Alpha={args.alpha}, Last_k={args.last_k}, Scale={args.scale}")
    
    # Process each input
    for i, user_input in enumerate(user_inputs):
        skeleton_id = user_input.get("skeleton_id", f"input_{i}")
        theme = user_input.get("theme", "Unknown")
        scenario = user_input["scenario"]
        
        print(f"\n[{i+1}/{len(user_inputs)}] {skeleton_id}")
        
        event_valences = {}
        
        # Choose valence keys
        if args.has_valence:
            valence_keys = VALENCES
        else:
            valence_keys = ["default"]
        
        for v_key in valence_keys:
            # Get event text
            if args.has_valence:
                if v_key not in user_input["event"]:
                    print(f"[WARN] Missing {v_key} in {skeleton_id}")
                    continue
                event_text = user_input["event"][v_key]
            else:
                event_text = user_input["event"]
            
            print(f"  Processing {v_key}...")
            emotion_texts = {}
            
            for emo in EMOS6:
                print(f"    Steering for {emo}...", end=" ", flush=True)
                
                # Create steerer for this emotion
                steerer = EmotionSteerer(
                    model, dirs, emo, layer_ids,
                    alpha=args.alpha,
                    last_k=args.last_k,
                    scale_mode=args.scale
                )
                
                try:
                    msgs = build_messages(scenario, event_text)
                    generated = generate_text(model, tok, msgs, max_new_tokens=args.max_new_tokens)
                    emotion_texts[emo] = generated
                    print(f"✓ ({len(generated)} chars)")
                except Exception as e:
                    emotion_texts[emo] = f"[ERROR] {str(e)}"
                    print(f"✗ {str(e)}")
                finally:
                    steerer.remove()
            
            event_valences[v_key] = emotion_texts
        
        # Save one JSON per sample
        result = {
            "skeleton_id": skeleton_id,
            "theme": theme,
            "scenario": scenario,
            "event_valences": event_valences,
            "parameters": {
                "layers": layer_ids,
                "alpha": args.alpha,
                "last_k": args.last_k,
                "scale": args.scale,
                "dtype": args.dtype,
                "direction_type": args.direction_type,
                "max_new_tokens": args.max_new_tokens,
                "sampling": "greedy",
                "model": "Qwen3-4B-Instruct"
            },
            "timestamp": int(time.time())
        }
        
        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
        
        print(f"  ✓ Saved result")
    
    print(f"\n[OK] All generations saved to {output_path}")
    print(f"Total samples processed: {len(user_inputs)}")

if __name__ == "__main__":
    main()

Emotion-Steered Generation Script (Qwen3-4B-Instruct)
Model: Qwen/Qwen3-4B-Instruct-2507
Layers: 15-25 (total 36 layers in Qwen)
Alpha: 8.0, Last_k: 1, Scale: rms
Direction type: mlp
[directions] Loaded 6 emotions, 36 layers, hidden dim 2560
[data] Loaded 160 inputs from /workspace/Various-Model/data/test_set.jsonl
Loading Qwen3-4B-Instruct model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded on cuda:0

[steering] Applying to layers: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
[steering] Alpha=8.0, Last_k=1, Scale=rms

[1/160] work_test_00
  Processing positive...
    Steering for anger... Found 36 layers in model
✓ (327 chars)
    Steering for sadness... Found 36 layers in model
✓ (382 chars)
    Steering for happiness... Found 36 layers in model
✓ (163 chars)
    Steering for fear... Found 36 layers in model
✓ (402 chars)
    Steering for surprise... Found 36 layers in model
✓ (83 chars)
    Steering for disgust... Found 36 layers in model
✓ (138 chars)
  Processing neutral...
    Steering for anger... Found 36 layers in model
✓ (343 chars)
    Steering for sadness... Found 36 layers in model
✓ (337 chars)
    Steering for happiness... Found 36 layers in model
✓ (162 chars)
    Steering for fear... Found 36 layers in model
✓ (433 chars)
    Steering for surprise... Found 36 layers in model
✓ (35 chars)
    Steering for disgust... Found 36 layers in model
✓ (

In [12]:
import sys

sys.argv = [
    "program",    
]

In [14]:
# scripts/03_emotion_elicited_generation_steer_based/2_label_steered_with_gpt.py
# -*- coding: utf-8 -*-
"""
GPT Labeling Script for Emotion-Steered Generated Text

Uses GPT-4o-mini to judge if generated text matches the target emotion.

Input: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/steered_outputs.jsonl
Output: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/labeled_results.jsonl
"""

import os, json, argparse, time, math, sys
from pathlib import Path
from typing import Dict, Any
from openai import OpenAI

# 6 emotions and 3 valences
EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]
VALENCES = ["positive", "neutral", "negative"]

# ========= OpenAI Client Setup =========
# Read API key from environment variable
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', None)
if not OPENAI_API_KEY:
    print("[WARNING] OPENAI_API_KEY not found. Please export it before running.")

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# GPT Prompts
SYSTEM = f'''
    You are a careful rater.
    Given a target emotion and a text,
    decide if the text's STYLE matches the target emotion among:
    {EMOTIONS}
    Focus on tone/attitude, not content valence.
'''

USER_TMPL = '''
    Target emotion: {emotion}
    Text:
    {text}
    Decide if the text's STYLE matches the target emotion.
    Return a compact JSON with keys exactly:
    {{
    "match": <0 or 1>,
    "reason": <short string>
    }}
'''

def extract_json_from_response(response: str) -> str:
    """
    Extract JSON content from GPT response, handling potential markdown blocks.
    """
    response = response.strip()
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    elif "```" in response:
        start = response.find("```") + 3
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    return response

def ask_llm(emotion: str, text: str, model: str, max_retries=4, backoff=1.8) -> Dict[str, Any]:
    """
    Call GPT to judge emotion matching with retry logic.
    """
    if client is None:
        return {"match": 0, "reason": "no-openai-client"}

    for i in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role":"system", "content": SYSTEM},
                    {"role":"user", "content": USER_TMPL.format(emotion=emotion, text=text)}
                ],
            )
            out = resp.choices[0].message.content
            print(f"      [{emotion}] GPT response captured.")

            json_str = extract_json_from_response(out)
            j = json.loads(json_str)

            # Ensure basic fields exist
            if "match" not in j: j = {"match": 0, "reason": "invalid-json"}
            if "reason" not in j: j["reason"] = "no-reason-provided"

            return j
        except Exception as e:
            print(f"      [{emotion}] Attempt {i+1} failed: {e}")
            if i == max_retries-1:
                return {"match": 0, "reason": f"error:{type(e).__name__}"}
            time.sleep(backoff ** i)

def get_processed_keys(output_file):
    """
    Get set of already processed sample keys to support resuming.
    """
    if not output_file.exists():
        return set()

    processed_keys = set()
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                obj = json.loads(line)
                key = f"{obj['skeleton_id']}___{obj.get('event_valence', 'neutral')}"
                processed_keys.add(key)
            except:
                continue
    return processed_keys
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="qwen3_4b_instruct_2507")
    parser.add_argument("--dataset_name", type=str, default="test_set")
    parser.add_argument("--gpt_model", type=str, default="gpt-4o-mini")
    args = parser.parse_args()

    model_name = args.model_name
    dataset_name = args.dataset_name

    input_jsonl = Path("/workspace/Various-Model/outputs") / model_name / "03_emotion_steered_generation" / dataset_name / "steered_outputs.jsonl"
    output_dir = Path("/workspace/Various-Model/outputs") / model_name / "03_emotion_steered_generation" / dataset_name
    output_jsonl = output_dir / "labeled_results.jsonl"

    if not input_jsonl.exists():
        print(f"[ERROR] Missing input file: {input_jsonl}")
        sys.exit(1)

    if client is None:
        print("[ERROR] OpenAI client not initialized.")
        sys.exit(1)

    print("GPT Labeling for Emotion-Steered Generated Text")
    print("=" * 60)

    processed_keys = get_processed_keys(output_jsonl)

    with open(input_jsonl, "r", encoding="utf-8") as fin, \
         open(output_jsonl, "a", encoding="utf-8") as fout:

        for line_num, line in enumerate(fin, 1):
            line = line.strip()
            if not line:
                continue

            try:
                item = json.loads(line)
            except:
                continue

            skeleton_id = item["skeleton_id"]
            sample_key = skeleton_id
            if sample_key in processed_keys:
                continue

            print(f"\n[{line_num}] Processing {skeleton_id}")

            event_valences = item.get("event_valences", {})
            valence_labels = {}

            # Loop over positive/neutral/negative
            for valence in VALENCES:
                if valence not in event_valences:
                    continue

                print(f"  Valence: {valence}")
                emotion_texts = event_valences[valence]
                emotion_labels = {}

                # Loop over emotions
                for emotion in EMOTIONS:
                    text = emotion_texts.get(emotion, "")

                    if not isinstance(text, str) or text.strip() == "":
                        label = {"match": 0, "reason": "empty-text"}
                    else:
                        label = ask_llm(emotion, text, args.gpt_model)

                    emotion_labels[emotion] = label
                    time.sleep(0.1)

                valence_labels[valence] = emotion_labels

            # Build final output
            result = {
                "skeleton_id": skeleton_id,
                "theme": item.get("theme", ""),
                "scenario": item.get("scenario", ""),
                "parameters": item.get("parameters", {}),
                "valence_labels": valence_labels,
                "event_valences": event_valences,
                "event": item.get("event", ""),
                "timestamp": int(time.time())
            }

            fout.write(json.dumps(result, ensure_ascii=False) + "\n")
            fout.flush()

    print(f"\n[OK] Saved labeled file to {output_jsonl}")


if __name__ == "__main__":
    main()

GPT Labeling for Emotion-Steered Generated Text

[1] Processing work_test_00
  Valence: positive
      [anger] GPT response captured.
      [sadness] GPT response captured.
      [happiness] GPT response captured.
      [fear] GPT response captured.
      [surprise] GPT response captured.
      [disgust] GPT response captured.
  Valence: neutral
      [anger] GPT response captured.
      [sadness] GPT response captured.
      [happiness] GPT response captured.
      [fear] GPT response captured.
      [surprise] GPT response captured.
      [disgust] GPT response captured.
  Valence: negative
      [anger] GPT response captured.
      [sadness] GPT response captured.
      [happiness] GPT response captured.
      [fear] GPT response captured.
      [surprise] GPT response captured.
      [disgust] GPT response captured.

[2] Processing work_test_01
  Valence: positive
      [anger] GPT response captured.
      [sadness] GPT response captured.
      [happiness] GPT response captured.
  

In [15]:
import sys

sys.argv = [
    "program",
]

In [16]:
# scripts/03_emotion_elicited_generation_steer_based/3_generate_accuracy_stats.py
# -*- coding: utf-8 -*-
"""
Emotion-Steered Generation Accuracy Statistics Script

Reads GPT labeling results, calculates accuracy by emotion and valence

Input: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/labeled_results.jsonl
Output: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/accuracy_stats.json
"""

import json, argparse
from pathlib import Path
from collections import defaultdict

def generate_accuracy_stats(output_dir: Path, dataset_name: str):
    """
    Generate accuracy statistics for specified dataset
    """

    dataset_dir = output_dir / dataset_name
    labeled_path = dataset_dir / "labeled_results.jsonl"
    stats_path = dataset_dir / "accuracy_stats.json"

    if not labeled_path.exists():
        print(f"[ERROR] Labeled results file not found: {labeled_path}")
        return None

    # Statistics dictionaries
    stats_by_emotion = defaultdict(lambda: {"total": 0, "matches": 0})
    stats_by_valence = defaultdict(lambda: {"total": 0, "matches": 0})

    total_pairs = 0
    total_matches = 0

    # Read labeled_results
    print(f"Reading {labeled_path}...")
    with open(labeled_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)

                # valence_labels contains positive, neutral, negative keys
                valence_labels = item.get("valence_labels", {})

                # Iterate through each valence
                for event_valence, emotion_labels in valence_labels.items():
                    # Count matching results for each emotion
                    for emotion, label in emotion_labels.items():
                        match = label.get("match", 0)

                        stats_by_emotion[emotion]["total"] += 1
                        stats_by_valence[event_valence]["total"] += 1
                        total_pairs += 1

                        if match == 1:
                            stats_by_emotion[emotion]["matches"] += 1
                            stats_by_valence[event_valence]["matches"] += 1
                            total_matches += 1

            except json.JSONDecodeError:
                continue

    # Calculate accuracy
    for emotion in stats_by_emotion:
        total = stats_by_emotion[emotion]["total"]
        matches = stats_by_emotion[emotion]["matches"]
        stats_by_emotion[emotion]["accuracy"] = round(matches / total * 100, 2) if total > 0 else 0.0

    for valence in stats_by_valence:
        total = stats_by_valence[valence]["total"]
        matches = stats_by_valence[valence]["matches"]
        stats_by_valence[valence]["accuracy"] = round(matches / total * 100, 2) if total > 0 else 0.0

    # Overall statistics
    overall_accuracy = round(total_matches / total_pairs * 100, 2) if total_pairs > 0 else 0.0

    # Build statistics result
    stats = {
        "dataset": dataset_name,
        "overall": {
            "total_pairs": total_pairs,
            "matches": total_matches,
            "accuracy": overall_accuracy
        },
        "by_emotion": dict(stats_by_emotion),
        "by_valence": dict(stats_by_valence)
    }

    # Save statistics file
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, stats_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="qwen3_4b_instruct_2507", help="Model folder name")
    parser.add_argument("--dataset", type=str, default="test_set", help="Dataset name (e.g., test_set, user_test)")
    parser.add_argument("--all", action="store_true", help="Process all datasets")
    args = parser.parse_args()

    # Determine output directory path
    output_dir = Path("/workspace/Various-Model/outputs") / args.model_name / "03_emotion_steered_generation"

    if not output_dir.exists():
        print(f"[ERROR] Output directory not found: {output_dir}")
        return

    # Determine datasets to process
    if args.all:
        # Auto-discover all dataset folders
        datasets = [d.name for d in output_dir.iterdir() if d.is_dir() and (d / "labeled_results.jsonl").exists()]
    elif args.dataset:
        datasets = [args.dataset]
    else:
        # Default: process all datasets with labeled_results.jsonl
        datasets = [d.name for d in output_dir.iterdir() if d.is_dir() and (d / "labeled_results.jsonl").exists()]

    if not datasets:
        print(f"[ERROR] No datasets with labeled_results.jsonl found in {output_dir}")
        return

    print("=" * 70)
    print("Generating Emotion Steering Accuracy Stats")
    print("=" * 70)

    for dataset_name in datasets:
        result = generate_accuracy_stats(output_dir, dataset_name)

        if result:
            stats, stats_path = result

            print(f"\n📊 {dataset_name.upper()} Dataset Statistics:")
            print(f"   File: {stats_path}")
            print(f"   Total Pairs: {stats['overall']['total_pairs']}")
            print(f"   Matches: {stats['overall']['matches']}")
            print(f"   Overall Accuracy: {stats['overall']['accuracy']}%")

            print(f"\n   By Emotion:")
            for emotion in sorted(stats['by_emotion'].keys()):
                e_stats = stats['by_emotion'][emotion]
                print(f"      {emotion:12s}: {e_stats['matches']:4d}/{e_stats['total']:4d} = {e_stats['accuracy']:6.2f}%")

            if stats['by_valence']:
                print(f"\n   By Valence:")
                for valence in sorted(stats['by_valence'].keys()):
                    v_stats = stats['by_valence'][valence]
                    print(f"      {valence:12s}: {v_stats['matches']:4d}/{v_stats['total']:4d} = {v_stats['accuracy']:6.2f}%")

    print("\n" + "=" * 70)
    print("✅ Statistics generation completed!")
    print("=" * 70)


if __name__ == "__main__":
    main()

Generating Emotion Steering Accuracy Stats
Reading /workspace/Various-Model/outputs/qwen3_4b_instruct_2507/03_emotion_steered_generation/test_set/labeled_results.jsonl...

📊 TEST_SET Dataset Statistics:
   File: /workspace/Various-Model/outputs/qwen3_4b_instruct_2507/03_emotion_steered_generation/test_set/accuracy_stats.json
   Total Pairs: 2880
   Matches: 2864
   Overall Accuracy: 99.44%

   By Emotion:
      anger       :  480/ 480 = 100.00%
      disgust     :  477/ 480 =  99.38%
      fear        :  480/ 480 = 100.00%
      happiness   :  480/ 480 = 100.00%
      sadness     :  480/ 480 = 100.00%
      surprise    :  467/ 480 =  97.29%

   By Valence:
      negative    :  959/ 960 =  99.90%
      neutral     :  950/ 960 =  98.96%
      positive    :  955/ 960 =  99.48%

✅ Statistics generation completed!
